# Ingest across four sources

[11b](11b_Bakery_Dataset_Agent_Ingest.ipynb) takes one export through [`optistock.ingest`](../docs/ingest.md) in depth. This notebook is the other half of the question: the same three agents, four different sources, read side by side. It is short on purpose — nothing here is fitted, and every cell reads a file.

Three of the four sources produce a canonical frame. One cannot, and the interesting part is *how* it fails: not with a crash, and not with a confident wrong answer, but with a required field the mapper declined to invent.

| Source | What makes it worth including |
|---|---|
| **French bakery** POS | The baseline from 11b: European decimal prices, returns as negative quantities. |
| **Online Retail II** (UCI) | The only source with a real buyer id, so the full canonical schema is reachable. |
| **Olist** order items | Validates cleanly and is still not what you want — the honest caveat case. |
| **dunnhumby** transactions | No calendar date exists anywhere in the table. The honest failure. |

> **Before running this notebook**, from the repository root:
> ```bash
> python scripts/prepare_bakery.py
> python scripts/prepare_online_retail.py
> python scripts/prepare_olist.py
> python scripts/prepare_dunnhumby.py
> ```
> Each needs `pip install optistock[ingest]` and a configured endpoint — see [docs/ingest.md](../docs/ingest.md#configuration), and run `optistock-ingest --check` first. Together they cost about $0.15 and a few large downloads. The notebook itself is model-free; any folder you skip is skipped here too.

> **Licence note:** none of the four datasets is committed. Each script writes to a git-ignored `data/` subdirectory.

> **Model behaviour is not reproducible.** The prose below describes what these runs actually did. A re-run can map a field differently or return a different verdict — the cells print what is on *your* disk, and where they disagree with the prose, your artifacts are right.

## 1. Four runs, side by side

In [1]:
import json
from pathlib import Path

import pandas as pd

# Everything read below is written by the scripts/prepare_*.py runs.
RUNS = {
    "bakery": Path("../data/bakery"),
    "online_retail": Path("../data/online_retail"),
    "olist": Path("../data/olist"),
    "dunnhumby": Path("../data/dunnhumby"),
}

summaries = {}
for name, folder in RUNS.items():
    path = folder / "ingest_summary.json"
    if path.exists():
        summaries[name] = json.loads(path.read_text(encoding="utf-8"))
    else:
        print(f"skipping {name}: run scripts/prepare_{name}.py to write {path}")

overview = pd.DataFrame(
    [
        {
            "rows": f"{s['n_rows']:,}",
            "rounds": s["rounds"],
            # .get: a summary written before the ok/error keys existed predates a re-run.
            "ok": s.get("ok", True),
            "verdict": s["review"]["verdict"],
            "issues": len(s["review"]["issues"]),
            "low confidence": ", ".join(s["low_confidence"]) or "-",
            "transforms that run": ", ".join(
                k for k, reason in s["transforms"].items() if reason is None
            )
            or "-",
        }
        for s in summaries.values()
    ],
    index=list(summaries),
)
overview

,rows,rounds,ok,verdict,issues,low confidence,transforms that run
bakery,"232,710",1,True,approve,0,-,"to_sales, to_returns"
online_retail,"1,044,421",1,True,approve,1,-,"to_sales, to_sales(split=True), to_clv, to_ret..."
olist,"112,650",1,True,approve,0,date,to_sales
dunnhumby,"2,581,266",1,False,approve,0,-,-


Three things are already visible without opening a single mapping.

**`ok` and `verdict` are different questions.** All four rows say `approve`, and one of them says `ok=False`. The verdict is agent 3's opinion of the mapping; `ok` is what [`validate`](../src/optistock/preprocessing/canonical.py) did to the frame that mapping produced. The loop only ever reads the verdict to decide whether to *retry* — it never lets the model declare success.

**No source needed a second round.** `rounds=1` everywhere: agent 3 approved on the first attempt each time, so the retry budget went unspent. That is the cheap path — 3 model calls rather than 6.

**What you can model is decided by the source, not the agent.** Only Online Retail II reaches `to_clv`, because only Online Retail II has a buyer identifier. No amount of prompting conjures one out of a till receipt.

## 2. The two that worked

### 2.1 Online Retail II — the full schema

Two years of a UK online gift retailer, 1.07M lines, and the only source here that carries `Customer ID`.

In [2]:
def artifacts(name: str) -> dict:
    """One run's summary, or a pointer to the script that would produce it."""
    if name not in summaries:
        raise FileNotFoundError(f"no artifacts for {name!r} — run scripts/prepare_{name}.py")
    return summaries[name]


def mapping_table(name: str) -> pd.DataFrame:
    """The mapping as agent 2 returned it: where each field came from, and how sure."""
    fields = artifacts(name)["mapping"]["mappings"]
    return pd.DataFrame(
        [
            {
                "strategy": m["strategy"],
                "from": m["source_column"] or m["expression"] or m["constant"] or "-",
                "declared format": m["date_format"] or m["numeric_format"] or "",
                "confidence": m["confidence"],
            }
            for m in fields
        ],
        index=[m["canonical_field"] for m in fields],
    )


def print_facts(name: str) -> None:
    """The relational view — usually the fastest way for a human to spot a bad mapping."""
    for key, value in artifacts(name)["facts"].items():
        print(f"  {key:34} {value}")


mapping_table("online_retail")

,strategy,from,declared format,confidence
order_id,direct,Invoice,,0.98
customer_id,direct,Customer ID,,0.90
date,direct,InvoiceDate,%Y-%m-%d %H:%M:%S,0.95
item_id,direct,StockCode,,0.95
quantity,direct,Quantity,,0.90
unit_price,direct,Price,,0.90
unit_cost,missing,-,,0.95


Six `direct` copies and one honest `missing` — no source in this notebook has a cost column, and declining is the right answer rather than a defect. `unit_cost` is what keeps `to_items` shut on all four.

The two numbers worth checking against the facts below: `orders_per_customer` should look like repeat retail rather than 1.0, and the date span should cover the two years the dataset claims.

In [3]:
print_facts("online_retail")

returns = pd.read_csv(RUNS["online_retail"] / "returns.csv")
runnable = [k for k, reason in artifacts("online_retail")["transforms"].items() if reason is None]
print(f"\n  {'returns held back':34} {len(returns):,}")
print(f"  {'transforms that run':34} {', '.join(runnable)}")

  n_order_lines                      1067371
  n_orders                           53628
  lines_per_order                    19.9
  n_customers                        5942
  orders_per_customer                7.55
  share_customers_with_one_order     0.2459
  n_items                            5305
  first_date                         2009-12-01 07:45:00
  last_date                          2011-12-09 12:50:00
  date_span_days                     738.2118
  n_distinct_dates                   604
  quantity_modal_value               1.0
  quantity_modal_share               0.2758
  quantity_is_integral               True
  quantity_min                       -80995.0
  quantity_max                       80995.0
  unit_price_median                  2.1
  unit_price_share_non_positive      0.0058

  returns held back                  22,950
  transforms that run                to_sales, to_sales(split=True), to_clv, to_returns


`7.55` orders per customer across 5,942 customers, spanning 738 days — a repeat-purchase order book, which is what makes `to_clv` meaningful here rather than merely runnable.

The 22,950 negative-quantity lines are the `'C'`-prefixed cancellation invoices. `to_canonical` captures them **before** dropping them, which is the only moment they exist — the demand panel is gross of returns, and without that capture the return rate would read 0% rather than "not recorded".

This run is also the one place agent 3 attached an issue to an `approve`. It objected that the *source* `Quantity` still contains negatives — then reasoned that row filtering is outside the mapper's four strategies and that no other column is a better candidate. It is describing something the deterministic half had already handled. A correct observation and a no-op, which is the shape a healthy review takes.

### 2.2 Olist — valid, and still not what you want

Only `olist_order_items_dataset.csv` is fed to the agents. The dataset is relational, and the purchase timestamp lives in a table the agents are not shown — which is exactly the situation this section exists to demonstrate.

In [4]:
display(mapping_table("olist"))

olist = artifacts("olist")
for field in ("date", "quantity"):
    note = next(
        m["note"] for m in olist["mapping"]["mappings"] if m["canonical_field"] == field
    )
    print(f"{field}: {note}\n")

print(f"reviewer verdict: {olist['review']['verdict']}")
print(f"reviewer issues : {olist['review']['issues'] or 'none'}")

,strategy,from,declared format,confidence
order_id,direct,order_id,,0.98
customer_id,missing,-,,0.90
date,direct,shipping_limit_date,%Y-%m-%d %H:%M:%S,0.40
item_id,direct,product_id,,0.95
quantity,constant,1,,0.75
unit_price,direct,price,,0.70
unit_cost,missing,-,,0.90


date: No true order-placed date exists in this table; shipping_limit_date (a logistics deadline) is used as the closest available timestamp, but it is not the actual order date.

quantity: Each row represents a single product line item; order_item_id is a per-order sequence index, not a quantity count, and no explicit quantity column exists.

reviewer verdict: approve
reviewer issues : none


The frame validates. Every required field is present and correctly typed, `to_sales` runs, and nothing downstream will complain. It is also wrong in a way no schema can detect: `date` is a **seller handover deadline**, not the moment anyone bought anything. A demand panel built on it is a panel of logistics deadlines.

Two details make this the most honest run of the four, and neither is the verdict:

* `confidence=0.4` on `date` — below the 0.5 threshold, so it surfaces in `low_confidence` in the overview table above, on a run that otherwise looks perfect;
* the note says it outright — *"a logistics deadline ... it is not the actual order date"*.

The reviewer approved with no issues, and that is defensible: `approve` means "no better mapping of **this table** exists", and none does. But it means the entire warning lives in a float and a sentence that nothing downstream reads. This is [the caveat in `docs/ingest.md`](../docs/ingest.md) — *a mapping can be well-formed and still wrong* — with a number attached.

`quantity` is the other thing to notice: there is no quantity column at all, one row is one unit, and the mapper said `constant` rather than reaching for `order_item_id`, which is a per-order sequence index and would have produced plausible nonsense.

## 3. The one that failed — dunnhumby

2.6M grocery lines from 2,500 households. Rich enough for CLV on paper: a stable household key, baskets, quantities, line totals. It has no calendar date anywhere.

In [5]:
dh = artifacts("dunnhumby")

profile = (RUNS["dunnhumby"] / "source_profile.txt").read_text(encoding="utf-8")
print("\n".join(ln for ln in profile.splitlines() if "DAY" in ln or "WEEK_NO" in ln))

date_field = next(m for m in dh["mapping"]["mappings"] if m["canonical_field"] == "date")
print(f"\nok       : {dh.get('ok')}")
print(f"date     : strategy={date_field['strategy']!r}  confidence={date_field['confidence']}")
print(f"           {date_field['note']}")
print(f"error    : {dh['error']}")
print(f"\nunmapped : {dh['mapping']['unmapped_source_columns']}")

- 'DAY' | dtype=int64 | null_rate=0% | unique=711 | samples: 1, 2, 3, 4, 5
- 'WEEK_NO' | dtype=int64 | null_rate=0% | unique=102 | samples: 1, 2, 3, 4, 5

ok       : False
date     : strategy='missing'  confidence=0.15
           DAY is a sequential integer counter (1..711) representing relative days since the start of data collection, not an actual calendar date or parseable date string. There is no epoch reference to convert it to a real datetime, so a true date cannot be derived from this source.
error    : orders is missing required column(s): ['date']. Present columns: ['order_id', 'customer_id', 'item_id', 'quantity', 'unit_price', 'DAY', 'STORE_ID', 'RETAIL_DISC', 'TRANS_TIME', 'WEEK_NO', 'COUPON_DISC', 'COUPON_MATCH_DISC'].

unmapped : ['DAY', 'STORE_ID', 'RETAIL_DISC', 'TRANS_TIME', 'WEEK_NO', 'COUPON_DISC', 'COUPON_MATCH_DISC']


`DAY` is an integer 1–711 counting from an origin the file never states, and `WEEK_NO` is the same information coarsened. `date` is a required canonical field, so there is no correct mapping to find — and the mapper said so, at `confidence=0.15`, rather than inventing an anchor. `DAY` survives as a pass-through covariate instead, which is where an integer day counter belongs.

Everything else mapped, and mapped well: `household_key → customer_id`, `SALES_VALUE / QUANTITY → unit_price` as an expression. 2.58M rows were built. Then `validate` named the missing column and the run ended `ok=False`, with every transform blocked on the same cause.

**This is the failure you want.** The alternative — mapping `DAY` straight into `date` — produces a frame that validates cleanly, reports 2.58M rows, gets approved, and places two and a half years of grocery shopping inside the first microsecond of 1 January 1970, because pandas reads a bare integer as nanoseconds. That frame is worse than no frame: nothing downstream would object to it, and the `to_clv` table it feeds would be confident, complete and entirely fictional.

The script still exited 0 and still wrote all five artifacts. A run that correctly determines a source cannot support the schema has produced a result, not an error.

In [6]:
# The chosen round and its transforms, verbatim from the run report.
report = (RUNS["dunnhumby"] / "ingest_report.txt").read_text(encoding="utf-8")
print(report[report.index("--- chosen ---") :])

--- chosen ---
ok=False  rows=2,581,266

orders is missing required column(s): ['date']. Present columns: ['order_id', 'customer_id', 'item_id', 'quantity', 'unit_price', 'DAY', 'STORE_ID', 'RETAIL_DISC', 'TRANS_TIME', 'WEEK_NO', 'COUPON_DISC', 'COUPON_MATCH_DISC'].

facts:
  n_order_lines = 2595732
  n_orders = 276484
  lines_per_order = 9.39
  n_customers = 2500
  orders_per_customer = 110.59
  share_customers_with_one_order = 0.0012
  n_items = 92339
  quantity_modal_value = 1.0
  quantity_modal_share = 0.7901
  quantity_is_integral = True
  quantity_min = 0.0
  quantity_max = 89638.0
  unit_price_median = 1.92
  unit_price_share_non_positive = 0.0017

--- transforms ---
  to_sales                 needs the required column(s) ['date'], which the schema demands
  to_sales(split=True)     needs the required column(s) ['date'], which the schema demands
  to_items                 needs the required column(s) ['date'], which the schema demands
  to_clv                   needs the require

## 4. What the four runs say about the loop

* **The gate is code, not the model.** Four `approve` verdicts, three usable frames. `ok` comes from `validate` alone, and dunnhumby is the run where the two disagree — which is the whole reason the deterministic half exists.
* **Honesty showed up as a number, not as a verdict.** Olist's problem is a `0.4`; dunnhumby's is `strategy='missing'` at `0.15`. Both were approved. On these four sources, reading only the verdict teaches you nothing.
* **So read `low_confidence` and `facts` every time.** `low_confidence` is what flagged Olist — the one frame here that validated cleanly and is still wrong. `facts` is the backstop for the mapping that goes wrong *without* saying so: `orders_per_customer = 1.0`, `date_span_days = 0`, a modal quantity of zero.
* **The source decides the ceiling.** `to_items` ran on none of the four, because no retail export ships cost data. `to_clv` ran on one, because one had a buyer id. That is a property of the data, and no agent changes it.
* **Read your own artifacts.** These four runs took one round each and about $0.15 in total. The next four may map a field differently. Everything above describes what happened once; it is not a specification.